# Pipeline de Ingesta: CSV → Landing S3
## TransCore Data Engineer Lab

Este notebook implementa el pipeline de ingestión de archivos CSV a la zona landing de S3 con metadatos de trazabilidad.

## 1. Configure Pipeline Parameters

In [ ]:
# Credenciales de AWS (reemplaza con tus propias credenciales)
import os
os.environ["AWS_ACCESS_KEY_ID"] = "<TU_ACCESS_KEY_ID>"
os.environ["AWS_SECRET_ACCESS_KEY"] = "<TU_SECRET_ACCESS_KEY>"

# Limpiar cualquier configuración de LocalStack
os.environ.pop("AWS_ENDPOINT_URL", None)
os.environ.pop("AWS_ENDPOINT", None)

# Configurar región de AWS explícitamente
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"
print("Configuración AWS establecida: AWS real, región eu-west-1")

## 2. Import Required Libraries

In [ ]:
import hashlib
import json
import logging
from datetime import datetime
from pathlib import Path

import boto3
import pandas as pd

# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("Librerías importadas correctamente")

In [ ]:
# Configuración
BUCKET_NAME = "<TU_BUCKET_NAME>"
SOURCE_FILE = "https://github.com/ricardoahumada/data-engineer-basics/raw/refs/heads/main/data/raw/partes_trabajo.csv"
S3_PREFIX = "landing/obra-lineal"

print(f"Bucket: {BUCKET_NAME}")
print(f"Source: {SOURCE_FILE}")
print(f"Prefix: {S3_PREFIX}")

## 3. Define IngestaPipeline Class

In [ ]:
class IngestaPipeline:
    """Pipeline de ingestión de archivos CSV a landing zone."""
    
    def __init__(self, bucket_name: str, s3_prefix: str):
        # Cliente S3 con región explícita para AWS real
        self.s3_client = boto3.client(
            "s3",
            region_name="eu-west-1"
        )
        self.bucket_name = bucket_name
        self.s3_prefix = s3_prefix
        self.ingesta_timestamp = datetime.utcnow().isoformat()
        self.ingesta_usuario = "data-engineer"
        
    def calcular_checksum(self, filepath: str) -> str:
        """Calcula SHA256 del archivo."""
        sha256_hash = hashlib.sha256()
        with open(filepath, "rb") as f:
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    
    def validar_formato(self, df: pd.DataFrame) -> bool:
        """Valida que el DataFrame contiene las columnas esperadas."""
        columnas_esperadas = [
            "id_parte", "id_equipo", "fecha_inicio", "fecha_fin",
            "horas_trabajo", "tipo_mantenimiento", "estado", "descripcion"
        ]
        columnas_faltantes = set(columnas_esperadas) - set(df.columns)
        
        if columnas_faltantes:
            logger.error(f"Columnas faltantes: {columnas_faltantes}")
            return False
        
        logger.info(f"Validación de formato: OK ({len(df.columns)} columnas)")
        return True
    
    def añadir_metadatos(self, df: pd.DataFrame, source_file: str) -> pd.DataFrame:
        """Añade metadatos de ingesta al DataFrame."""
        df["_ingesta_timestamp"] = self.ingesta_timestamp
        df["_ingesta_usuario"] = self.ingesta_usuario
        df["_source_file"] = source_file
        logger.info(f"Metadatos añadidos: timestamp={self.ingesta_timestamp}")
        return df
    
    def generar_s3_key(self) -> str:
        """Genera la clave única S3 con estructura de directorios por fecha."""
        ahora = datetime.now()
        timestamp_str = ahora.strftime("%H%M%S")
        fecha_str = ahora.strftime("%Y/%m/%d")
        
        # Nombre archivo con timestamp
        filename = f"partes_trabajo_{timestamp_str}.csv"
        
        # Clave S3 completa con particionamiento Hive-style
        s3_key = f"{self.s3_prefix}/año={fecha_str.split('/')[0]}/mes={fecha_str.split('/')[1]}/dia={fecha_str.split('/')[2]}/{filename}"
        
        return s3_key
    
    def ejecutar(self, source_filepath: str) -> dict:
        """Ejecuta el pipeline completo de ingestión."""
        logger.info(f"Iniciando pipeline de ingestión: {source_filepath}")
        
        # 1. Leer archivo fuente
        logger.info("Paso 1: Leyendo archivo fuente...")
        df = pd.read_csv(source_filepath)
        logger.info(f"  - Registros leídos: {len(df)}")
        
        # 2. Validar formato
        logger.info("Paso 2: Validando formato...")
        if not self.validar_formato(df):
            raise ValueError("Validación de formato fallida")
        
        # 3. Añadir metadatos
        logger.info("Paso 3: Añadiendo metadatos de ingestión...")
        df = self.añadir_metadatos(df, Path(source_filepath).name)
        
        # 4. Guardar temporalmente con metadatos (compatible Windows/Linux)
        import tempfile
        temp_file = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False).name
        df.to_csv(temp_file, index=False)
        
        # 5. Calcular checksum
        logger.info("Paso 4: Calculando checksum SHA256...")
        checksum = self.calcular_checksum(temp_file)
        logger.info(f"  - Checksum: {checksum[:16]}...")
        
        # 6. Generar clave S3
        s3_key = self.generar_s3_key()
        logger.info(f"Paso 5: Subiendo a S3: s3://{self.bucket_name}/{s3_key}")
        
        # 7. Subir a S3
        self.s3_client.upload_file(temp_file, self.bucket_name, s3_key)
        logger.info("  - Upload completado")
        
        # 8. Guardar metadata de ingestión
        metadata = {
            "s3_key": s3_key,
            "bucket": self.bucket_name,
            "checksum_sha256": checksum,
            "ingesta_timestamp": self.ingesta_timestamp,
            "ingesta_usuario": self.ingesta_usuario,
            "source_file": Path(source_filepath).name,
            "num_registros": len(df),
            "columnas": list(df.columns)
        }
        
        metadata_key = s3_key.replace(".csv", "_metadata.json")
        self.s3_client.put_object(
            Bucket=self.bucket_name,
            Key=metadata_key,
            Body=json.dumps(metadata, indent=2),
            ContentType="application/json"
        )
        logger.info(f"Paso 6: Metadata guardada en {metadata_key}")
        
        # Limpieza
        Path(temp_file).unlink()
        
        return metadata

print("IngestaPipeline class definida correctamente")

## 4. Execute the Pipeline

In [ ]:
# Instanciar y ejecutar el pipeline
pipeline = IngestaPipeline(BUCKET_NAME, S3_PREFIX)
result = pipeline.ejecutar(SOURCE_FILE)
logger.info(f"Pipeline completado exitosamente. S3 Key: {result['s3_key']}")

# Mostrar resultado
print(f"\nPipeline completado:")
print(f"  - S3 Key: {result['s3_key']}")
print(f"  - Checksum: {result['checksum_sha256'][:16]}...")
print(f"  - Registros: {result['num_registros']}")